# 04 — Bayesian fitting with DREAM (Student workbook)

**Time:** 60–75 minutes (advanced extension)
**Question:** what can a posterior distribution tell us that a single best fit
cannot?

Earlier notebooks returned one optimum and a local uncertainty estimate. Here
you will sample a distribution of plausible parameter combinations using the
DREAM Markov chain Monte Carlo algorithm.

In [ ]:
from pathlib import Path
import sys

ROOT = next(path for path in [Path.cwd(), *Path.cwd().parents] if (path / 'pyproject.toml').exists())
sys.path.insert(0, str(ROOT / 'src'))

import matplotlib.pyplot as plt
import numpy as np
import sans_fitter
from sans_fitter import SANSFitter

DATA = ROOT / 'data'
print(f'Using sans_fitter from {Path(sans_fitter.__file__).resolve()}')
%matplotlib inline


In [ ]:
filename = DATA / 'sphere_reference.csv'
values = np.genfromtxt(filename, delimiter=',', names=True)
q, intensity, uncertainty = values['Q'], values['I'], values['dI']


## 1 — From bounds to priors

Bayes' theorem combines a likelihood with prior information:

$$
p(\theta \mid D) \propto p(D \mid \theta)\,p(\theta).
$$

For this workshop, `set_param(..., min=a, max=b, vary=True)` defines a
**uniform prior** between `a` and `b`: every value inside is initially equally
plausible and values outside are forbidden. This is not merely an optimizer
safety rail. Bounds must therefore be scientifically defensible.

Configure the same sphere model as before. Keep known SLD values fixed and use
bounded priors for radius, scale, and background.

In [ ]:
fitter = SANSFitter()
fitter.load_data(str(filename))

# Select a sphere model and inspect its parameter names. Keep both SLDs fixed,
# then give radius, scale, and background proper bounded uniform priors.
# Avoid bounds that are so broad that the short classroom chain cannot explore.

# YOUR CODE HERE
raise NotImplementedError('Configure the sphere model and its priors')


## 2 — Sampling with DREAM

`fit_bayesian` uses several controls:

- `samples`: approximate posterior sampling budget;
- `burn`: early generations discarded while chains approach high probability;
- `thin`: retain every nth draw;
- `pop`: population size multiplier for DREAM's parallel chains.

A larger budget is not automatically trustworthy. Chains must mix, explore the
same region, and be insensitive to starting values. The short run below is
chosen for class time and is **exploratory, not publication quality**.

In [ ]:
# For a classroom-scale exploratory run, use:
#   samples=1500, burn=100, thin=1, pop=10
# Store the result and plot the best fit with normalized residuals.

# YOUR CODE HERE
raise NotImplementedError('Run DREAM and inspect the fitted curve')


## 3 — Posterior data and convergence

`get_posterior()` returns:

- `labels` and a two-dimensional `samples` array;
- 68% and 95% credible intervals;
- separate `chains` where available;
- R-hat diagnostics comparing within-chain and between-chain variation.

R-hat near 1 is desirable. Values above roughly 1.05 are a warning to run
longer, revisit priors, or investigate a multimodal posterior. Trace plots
should look stationary and well mixed rather than drifting or separating.

In [ ]:
posterior = fitter.get_posterior()
print('Parameters:', posterior.labels)
print('Combined sample shape:', posterior.samples.shape)

# YOUR CODE HERE
# Print each 95% credible interval and R-hat, then call `plot_trace()`.
# Identify any parameter for which the chains have not mixed well.
raise NotImplementedError('Inspect convergence and credible intervals')


## 4 — Marginal distributions and correlations

A marginal distribution shows uncertainty in one parameter after averaging
over the others. A pair plot and correlation matrix reveal combinations that
the data constrain together. For example, particle contrast, concentration,
and scale can compensate for one another.

In [ ]:
# Use:
#   plot_param_distribution('radius')
#   plot_posterior_pairs()
#   plot_param_correlations()
# Explain one correlation physically in a Markdown cell.

# YOUR CODE HERE
raise NotImplementedError('Explore marginal and joint posterior structure')


## 5 — Posterior predictive checking

Parameter intervals are not the final scientific target. A posterior
predictive check propagates sampled parameters through the scattering model.
The resulting band asks whether plausible parameter sets reproduce the
observations across q, and where predictions remain uncertain.

In [ ]:
# Plot a 95% posterior predictive band with 40 individual draws.
# Decide whether uncertainty is equally large across the measured q range.

# YOUR CODE HERE
raise NotImplementedError('Create and interpret a posterior predictive check')


## 6 — Interpretation

Write a short report addressing:

1. Which parameter is best constrained, and what feature of the curve does it control?
2. Which parameters are correlated, and why?
3. Do the traces and R-hat values support convergence?
4. Where is the predictive band widest?
5. What would you change before publishing these intervals?

In [ ]:
# Summarize what DREAM adds beyond the point estimate. Refer to the
# credible intervals, trace behavior, parameter correlations, and predictive band.
interpretation = 'Write your evidence-based interpretation here.'
print(interpretation)


## Optional export

The raw chain can be analyzed with other tools:

```python
posterior.save_posterior_csv('sphere_posterior.csv')
fitter.save_results('sphere_bayesian_fit.csv')
```

Try changing one prior range or restricting q. Rerun the sampler and explain
whether the posterior changed because of the data or because of the prior.